# Wireless Networking and Mobile Computing

### Prerequisites

This Jupyter Notebook has been tested with Visual Studio Code, running in a local Python environment.

**Installs**

In [ ]:
# --- Installs: ensure required packages are available ---
%pip install numpy pandas matplotlib --upgrade pip --quiet

**Imports**

In [ ]:
# --- Imports: libraries used in this notebook ---
import re
import xml.etree.ElementTree as ET
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import to_rgb, to_hex
from matplotlib.lines import Line2D


**Declarations**

Set the variables below and run the generation cell. 

- `scenario` is the path to the `.xml` scenario file that will be simulated.

In [ ]:
# --- Declarations: editable variables for students ---
scenario = "./jemula-project/jemula802/scenarios/ETH_Course/WNMC_Resource_Sharing.xml"

### 1 Simulation Results

#### 1.1. Folder Location
We dynamically extract the folder location from the property `Path2Results`.

In [ ]:
# --- Locate Results Folder ---
tree = ET.parse(scenario)
root = tree.getroot()
path_elem = root.find(".//JE802StatEval")
path2results = path_elem.get("Path2Results") if path_elem is not None else None

if not path2results:
    raise ValueError("Path2Results not found in scenario XML")

results_dir = Path(path2results)
print(f"Found results folder: {results_dir}")
if not results_dir.is_dir():
    raise FileNotFoundError(
        f"Results folder {results_dir} does not exist. Run Jemula802_Run.ipynb first."
    )
print("\nCSV files:")
for result_file in sorted(results_dir.glob("*.csv")):
    print("  -", result_file.name)


### 2 Statistical Evaluation

#### 2.1 Plotting Functions

In [ ]:
def _time_column(frame):
    for column in frame.columns:
        name = str(column).strip().lower()
        if name == "time" or name.startswith(("time[", "time [")):
            return column
    return None


def _select_measurement_column(frame, metric, path):
    preferred_columns = {
        "throughput": "thrp last interval [mb/s]",
        "offer": "offer last interval [mb/s]",
        "delay": "avrg last interval [ms]",
    }
    preferred = preferred_columns[metric]
    columns = {str(name).strip().lower(): name for name in frame.columns}
    if preferred in columns:
        return columns[preferred]

    time_column = _time_column(frame)
    candidates = [
        name for name in frame.columns
        if name != time_column
        and pd.to_numeric(frame[name], errors="coerce").notna().any()
    ]
    if len(candidates) == 1:
        return candidates[0]
    if not candidates:
        raise ValueError(f"No numeric {metric} column found in {path}.")
    raise ValueError(
        f"Several numeric columns are available in {path}; expected '{preferred}', "
        f"found {candidates}."
    )


def metric_time_series(frame, metric, path="Jemula result"):
    """Return aligned time [ms] and measurement arrays."""
    if metric not in ("throughput", "offer", "delay"):
        raise ValueError(f"Unknown metric {metric!r}.")

    time_column = _time_column(frame)
    if time_column is None:
        raise ValueError(
            f"No time column found in {path}; expected Jemula's 'time[ms]' field."
        )
    measurement_column = _select_measurement_column(frame, metric, path)
    times = pd.to_numeric(frame[time_column], errors="coerce").to_numpy(dtype=float)
    values = pd.to_numeric(frame[measurement_column], errors="coerce").to_numpy(dtype=float)
    valid = np.isfinite(times) & np.isfinite(values)
    if "last interval" in str(measurement_column).strip().lower():
        # The first last-interval row has no preceding interval.
        valid &= times > 0

    times, values = times[valid], values[valid]
    if not len(values):
        raise ValueError(f"No finite {metric} samples found in {path}.")
    return times, values


# --- Plotting & Helper Functions ---
def resolve_result_path(path):
    result_path = Path(path)
    if not result_path.is_absolute() and not result_path.is_file():
        result_path = results_dir / result_path
    return result_path


def safe_read_csv(path):
    if path is None or path in (..., ""):
        return None
    result_path = resolve_result_path(path)
    if not result_path.is_file():
        raise FileNotFoundError(f"CSV file not found: {result_path}")
    try:
        frame = pd.read_csv(result_path, encoding="utf-8-sig")
    except pd.errors.EmptyDataError:
        raise ValueError(f"CSV file is empty: {result_path}") from None
    if frame.empty:
        raise ValueError(f"CSV file has a header but no data rows: {result_path}")
    frame.columns = [str(column).strip() for column in frame.columns]
    return frame


def plot_all_thrp_vs_offer(offer_file=None, thrp_files=None, figsize=(10, 5)):
    """Plot Jemula throughput intervals for each station plus total offer/throughput."""
    csv_paths = sorted(results_dir.glob("*.csv"))
    if thrp_files is None:
        thrp_files = [
            path for path in csv_paths
            if path.name.lower().startswith('thrp_') and 'total' not in path.name.lower()
        ]
    else:
        thrp_files = [resolve_result_path(path) for path in thrp_files]

    if offer_file is not None:
        offer_path = resolve_result_path(offer_file)
    else:
        offer_candidates = [path for path in csv_paths if 'offer' in path.name.lower()]
        total_candidates = [path for path in offer_candidates if 'total' in path.name.lower()]
        offer_path = (total_candidates or offer_candidates or [None])[0]

    fig, ax = plt.subplots(figsize=figsize)
    ax.set_facecolor((.2, 0.2, 0.3))
    ax.grid(color="#AAAAAA", linewidth=.5, linestyle='-', alpha=1)
    ax.set_xlabel('Simulated Time [s]')
    ax.set_ylabel('Throughput [Mb/s]')
    ax.minorticks_on()

    color_group_1 = '#66c2a5'
    color_group_2 = '#fc8d62'
    color_other = '#8da0cb'
    entries = []
    for path in thrp_files:
        match = re.search(r'thrp_SA(\d+)_', path.name, flags=re.I)
        station = int(match.group(1)) if match else None
        group = 'g1' if station is not None and 1 <= station <= 3 else (
            'g2' if station is not None and 4 <= station <= 6 else 'other'
        )
        entries.append((path, station, group))

    groups = {
        'g1': [entry for entry in entries if entry[2] == 'g1'],
        'g2': [entry for entry in entries if entry[2] == 'g2'],
        'other': [entry for entry in entries if entry[2] == 'other'],
    }

    def shade(hex_color, amount):
        red, green, blue = to_rgb(hex_color)
        return to_hex((red + (1 - red) * amount,
                       green + (1 - green) * amount,
                       blue + (1 - blue) * amount))

    bases = {'g1': color_group_1, 'g2': color_group_2, 'other': color_other}
    color_map = {}
    for group_name, members in groups.items():
        for index, entry in enumerate(members):
            amount = 0.0 if len(members) == 1 else 0.6 * index / (len(members) - 1)
            color_map[entry[0]] = shade(bases[group_name], amount)

    max_time_ms = 0.0
    plotted_stations = 0
    for path, station, _group in entries:
        frame = safe_read_csv(path)
        time_ms, values = metric_time_series(frame, 'throughput', path=str(path))
        label = f'Throughput Station {station:02d} [Mb/s]' if station is not None else path.stem
        ax.plot(time_ms / 1000.0, values, '-', color=color_map[path], linewidth=2, alpha=.95, label=label)
        max_time_ms = max(max_time_ms, float(np.max(time_ms)))
        plotted_stations += 1

    total_candidates = [
        path for path in csv_paths
        if path.name.lower().startswith('total_thrp') or 'total_thrp' in path.name.lower()
    ]
    if total_candidates:
        total_path = sorted(total_candidates)[0]
        frame = safe_read_csv(total_path)
        time_ms, values = metric_time_series(frame, 'throughput', path=str(total_path))
        ax.plot(time_ms / 1000.0, values, '-', color='#666666', linewidth=4,
                alpha=.95, label='Total throughput [Mb/s]')
        max_time_ms = max(max_time_ms, float(np.max(time_ms)))

    if offer_path is not None:
        offer_df = safe_read_csv(offer_path)
        time_ms, values = metric_time_series(offer_df, 'offer', path=str(offer_path))
        ax.plot(time_ms / 1000.0, values, '-', color='red', linewidth=3,
                label='Total offer [Mb/s]')
        max_time_ms = max(max_time_ms, float(np.max(time_ms)))

    if plotted_stations == 0 and not total_candidates and offer_path is None:
        raise ValueError(f"No throughput or offer CSV files found in {results_dir}.")
    if max_time_ms > 0:
        ax.set_xlim(0, max_time_ms / 1000.0 * 1.05)

    if len(entries) + len(total_candidates) + (offer_path is not None) <= 20:
        ax.legend(fancybox=True, framealpha=1, facecolor=(.6, .61, .8), fontsize='small')
    else:
        legend_items = [
            Line2D([0], [0], color='red', lw=3, label='Total offer'),
            Line2D([0], [0], color='#666666', lw=4, label='Total throughput'),
            Line2D([0], [0], color=color_group_1, lw=2, label='Stations SA1-SA3'),
            Line2D([0], [0], color=color_group_2, lw=2, label='Stations SA4-SA6'),
            Line2D([0], [0], color=color_other, lw=2, label='Other stations'),
        ]
        ax.legend(handles=legend_items, fancybox=True, framealpha=1,
                  facecolor=(.6, .61, .8), fontsize='small')

    fig.tight_layout()
    return fig, ax


#### 2.2 Total System Plots

In [ ]:
# --- Plot Total Results ---
fig, ax = plot_all_thrp_vs_offer()